# 📓 Notebook 4: Đánh Giá Định Lượng & Phân Tích Trường Hợp Thất Bại
**Căn cứ tài liệu:** `docs/KHUNG_NOI_DUNG_TONG_QUAT_DO_AN_XLAS.md`  
* **Quy trình chuẩn Bộ môn:**  
  * **Bước 5 — Xây dựng Baseline & Thực nghiệm so sánh (Comparative Experiments)**
  * **Bước 6 — Đánh giá định lượng & Phân tích trường hợp thất bại (Evaluation & Error Analysis)**
* **Ánh xạ Báo cáo đồ án:** **Mục 6 — Kết quả và đánh giá**  
* **Học phần:** Xử lý ảnh số (2026–2027) — Khoa CNTT — Trường ĐH Công Thương TP.HCM (HUIT)  
* **Nhóm thực hiện:** Hà Mạnh Trường (2045240326) & Huỳnh Tâm Trí (2045240315) — *Trí phụ trách chính*

---

## 🎯 1. Bộ đo định lượng y khoa chuẩn mực (Theo Khung Tổng Quát Mục II.6 & IV)
1. **Dice Similarity Coefficient (DSC / F1):** $	ext{Dice} = rac{2|P \cap G|}{|P| + |G|}$
2. **Jaccard Index (IoU):** $	ext{IoU} = rac{|P \cap G|}{|P \cup G|} = rac{	ext{Dice}}{2 - 	ext{Dice}}$
3. **Độ nhạy (Sensitivity / Recall):** $	ext{Sensitivity} = rac{	ext{TP}}{	ext{TP} + 	ext{FN}}$ (Đo khả năng không bỏ sót pixel u)
4. **Độ đặc hiệu (Specificity):** $	ext{Specificity} = rac{	ext{TN}}{	ext{TN} + 	ext{FP}}$ (Đo khả năng không nhận nhầm não lành)
5. **Độ trễ tính toán:** Thời gian xử lý trung bình trên từng khung hình/ảnh (Latency ms).


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))
from src.preprocessing.skull_strip import strip_skull
from src.segmentation.watershed import segment_watershed
from src.segmentation.region_growing import segment_region_growing
from src.evaluation.benchmark import run_benchmark
from src.evaluation.metrics import calculate_dice, calculate_iou, calculate_sensitivity, calculate_specificity

img_dir = "../data/sample/images"
mask_dir = "../data/sample/masks"
print("Sẵn sàng thực thi benchmark trên toàn bộ 20 mẫu ảnh TCGA-LGG...")


## 🚀 2. Chạy Thực Nghiệm Đánh Giá Hàng Loạt (Batch Benchmark) 20 Mẫu


In [ ]:
def predict_ws(img):
    stripped, _ = strip_skull(img)
    return segment_watershed(stripped, dist_thresh_ratio=0.4)

def predict_rg(img):
    stripped, _ = strip_skull(img)
    return segment_region_growing(stripped, tolerance=15)

# Chạy benchmark
df_ws = run_benchmark(img_dir, mask_dir, predict_ws, method_name="Watershed (Baseline)")
df_rg = run_benchmark(img_dir, mask_dir, predict_rg, method_name="Region Growing (Đối chứng)")

print(f"Đã hoàn thành đánh giá {len(df_ws)} mẫu cho Watershed và {len(df_rg)} mẫu cho Region Growing.")


## 📊 3. Bảng Tổng Hợp Kết Quả Định Lượng (Mean ± Std)
Đây là bảng số liệu trọng tâm để đưa vào Mục 6 của Báo cáo đồ án:


In [ ]:
summary_df = pd.DataFrame({
    "Chỉ Số Đánh Giá Định Lượng": [
        "1. Dice Similarity Coefficient (DSC)",
        "2. Jaccard Index (IoU)",
        "3. Sensitivity (Độ nhạy - Bắt đúng u)",
        "4. Specificity (Độ đặc hiệu - Tránh nhầm)",
        "5. Thời gian xử lý trung bình (Latency)"
    ],
    "Phương án A: Watershed (Baseline)": [
        f"{df_ws['dice'].mean():.4f} ± {df_ws['dice'].std():.4f}",
        f"{df_ws['iou'].mean():.4f} ± {df_ws['iou'].std():.4f}",
        f"{df_ws['sensitivity'].mean():.4f} ± {df_ws['sensitivity'].std():.4f}",
        f"{df_ws['specificity'].mean():.4f} ± {df_ws['specificity'].std():.4f}",
        f"{df_ws['latency_ms'].mean():.2f} ± {df_ws['latency_ms'].std():.2f} ms"
    ],
    "Phương án B: Region Growing (Đối chứng)": [
        f"{df_rg['dice'].mean():.4f} ± {df_rg['dice'].std():.4f}",
        f"{df_rg['iou'].mean():.4f} ± {df_rg['iou'].std():.4f}",
        f"{df_rg['sensitivity'].mean():.4f} ± {df_rg['sensitivity'].std():.4f}",
        f"{df_rg['specificity'].mean():.4f} ± {df_rg['specificity'].std():.4f}",
        f"{df_rg['latency_ms'].mean():.2f} ± {df_rg['latency_ms'].std():.2f} ms"
    ]
})

summary_df


## 📉 4. Biểu Đồ Cột So Sánh Điểm Dice Trên Từng Ca Bệnh
Trực quan hóa sự khác biệt giữa 2 thuật toán trên toàn bộ tập thử nghiệm:


In [ ]:
x = np.arange(len(df_ws))
width = 0.38

fig, ax = plt.subplots(figsize=(16, 6))
rects1 = ax.bar(x - width/2, df_ws['dice'], width, label='Phương án A: Watershed (Baseline)', color='#3B82F6', alpha=0.85)
rects2 = ax.bar(x + width/2, df_rg['dice'], width, label='Phương án B: Region Growing (Đối chứng)', color='#10B981', alpha=0.85)

ax.set_ylabel('Điểm Dice Similarity Coefficient', fontsize=12)
ax.set_title('So Sánh Điểm Dice Score Giữa Watershed và Region Growing Trên 20 Ca Bệnh TCGA-LGG', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f"Ca {i+1:02d}" for i in range(len(df_ws))], rotation=45)
ax.legend(fontsize=12)
ax.set_ylim(0, 1.05)
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


## ⚠️ 5. Phân Tích Trường Hợp Thất Bại (Failure Cases Analysis)
Theo yêu cầu bắt buộc của Khung tổng quát (Mục II.6): Phải chỉ ra và lý giải nguyên nhân sai lệch.


In [ ]:
# Tìm các ca có điểm Dice thấp nhất
df_compare = pd.DataFrame({
    'sample_id': df_ws['sample_id'],
    'dice_ws': df_ws['dice'],
    'dice_rg': df_rg['dice']
})
df_compare['min_dice'] = df_compare[['dice_ws', 'dice_rg']].min(axis=1)
worst_cases = df_compare.sort_values(by='min_dice').head(3)

print("3 Ca bệnh khó nhất / có điểm Dice thấp nhất:")
print(worst_cases)

# Hiển thị trực quan ca lỗi nặng nhất
worst_id = worst_cases.iloc[0]['sample_id']
w_img = cv2.imread(f"../data/sample/images/{worst_id}.png", cv2.IMREAD_GRAYSCALE)
w_gt = cv2.imread(f"../data/sample/masks/{worst_id}_mask.png", cv2.IMREAD_GRAYSCALE)
w_stripped, _ = strip_skull(w_img)
w_pred_ws = segment_watershed(w_stripped)
w_pred_rg = segment_region_growing(w_stripped)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(w_img, cmap='gray'); axes[0].set_title(f"Ca Lỗi: {worst_id}"); axes[0].axis('off')
axes[1].imshow(w_gt, cmap='hot'); axes[1].set_title("Ground Truth Bác Sĩ"); axes[1].axis('off')
axes[2].imshow(w_pred_ws, cmap='Blues'); axes[2].set_title(f"Watershed (Dice={worst_cases.iloc[0]['dice_ws']:.3f})"); axes[2].axis('off')
axes[3].imshow(w_pred_rg, cmap='Greens'); axes[3].set_title(f"Region Growing (Dice={worst_cases.iloc[0]['dice_rg']:.3f})"); axes[3].axis('off')
plt.suptitle("Phân Tích Ca Thất Bại Điển Hình (Failure Case)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("""
NHẬN XÉT NGUYÊN NHÂN THẤT BẠI (ĐƯA VÀO BÁO CÁO):
1. Khối u có độ tương phản quá thấp so với chất xám xung quanh (ranh giới bờ mờ).
2. Kích thước u quá nhỏ khiến bản đồ khoảng cách Euclidean (Distance Transform) bị suy biến đỉnh cực đại.
3. Hướng khắc phục: Cần kết hợp thêm thông tin kết cấu GLCM hoặc sử dụng ngưỡng thích ứng cục bộ.
""")
